# 1. CLEAN WAHIS

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import pycountry 

In [ ]:
# Import the data
df = pd.read_excel("data/infur_20260401.xlsx", sheet_name=1) 

In [ ]:
df.columns = df.columns.str.lower()

In [ ]:
df = df.loc[df["region"]=="Europe"]

In [ ]:
# Define a function to identify the first outbreak of WN 
def clean_wahis_outbreak_disease(df: pd.DataFrame, disease: str) -> pd.DataFrame:
    df = df.copy()
    df = df.loc[df["disease_eng"] == disease]

    if disease == "West Nile Fever":
        def recategorize_species(species):
            if species == "Equidae (dom)":
                return "horse"
            elif species == "Dogs":
                return "dog"
            else:
                return "bird"
        
        df["species"] = df["species"].apply(recategorize_species)

    df["reporting_date"] = pd.to_datetime(df["reporting_date"], errors="coerce").dt.normalize()
    df["outbreak_start_date"] = pd.to_datetime(df["outbreak_start_date"], errors="coerce").dt.normalize()

    first_reporting = (
        df.groupby(["outbreak_id", "iso_code", "country", "level1_name", "species"])["reporting_date"]
        .min()
        .reset_index()
        .rename(columns={"reporting_date": "first_reporting_date"})
    )

    out_core = (
        df[["outbreak_id",
            "iso_code",
            "country",
            "level1_name",
            "species", 
            "outbreak_start_date"
        ]]
        .drop_duplicates()
    )

    clean = out_core.merge(
        first_reporting, 
        on=["outbreak_id", "country", "iso_code", "level1_name", "species"], 
        how="left"
    )
    
    clean["reporting_delay_days"] = (
        clean["first_reporting_date"] - clean["outbreak_start_date"]
    ).dt.days
    
    clean = clean.sort_values("outbreak_start_date")
    
    # Identify the indices of the first outbreak for each group
    group_cols = ["iso_code", "country", "level1_name", "species"]
    idx = clean.groupby(group_cols)["outbreak_start_date"].idxmin()
    first_outbreaks = clean.loc[idx].reset_index(drop=True)

    return first_outbreaks

In [ ]:
df_wn = clean_wahis_outbreak_disease(df, disease="West Nile Fever")

In [ ]:
# Filter events occurring from 2018 onwards with reporting delay greater than 0, given that EIOS has been consistently operational since November 2017.
df_wn=df_wn.loc[(df_wn["outbreak_start_date"]>= "2018-01-01") & (df_wn["reporting_delay_days"]>0)]
len(df_wn)

In [ ]:
df_wn["country"].unique()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 12))

sns.countplot(
    data=df_wn,
    y="country",
    order=df_wn["country"].value_counts().index,
    color="purple",
    ax=ax
)

# Add count labels
for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%d",
        padding=3,
        fontsize=9
    )


ax.set_title(
    "Outbreaks by country",
    fontsize=15,
    weight="bold"
)
ax.set_xlabel("Count", fontsize=11)
ax.set_ylabel("Country", fontsize=11)

ax.tick_params(axis="both", labelsize=10)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

plt.savefig(
    "results/outbreaks_by_country.pdf",
    dpi=600,
    bbox_inches="tight"
)
plt.savefig(
    "results/outbreaks_by_country.tiff",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 12))


df_delay = (
    df_wn.groupby("country")["reporting_delay_days"]
    .mean()
    .sort_values()
    .reset_index()
)

sns.scatterplot(
    data=df_delay,
    x="reporting_delay_days",
    y="country",
    color="purple",
    s=80,
    ax=ax
)

# Add value labels
for _, row in df_delay.iterrows():
    ax.text(
        row["reporting_delay_days"] + 2.5,
        row["country"],
        f"{row['reporting_delay_days']:.1f}",
        va="center",
        fontsize=9
    )

ax.set_title(
    "Average reporting delay by country",
    fontsize=15,
    weight="bold"
)

ax.set_xlabel("Days", fontsize=11)
ax.set_ylabel("Country", fontsize=11)

ax.tick_params(axis="both", labelsize=10)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.margins(x=0.15)

plt.tight_layout()

plt.savefig(
    "results/reporting_delay_by_country.pdf",
    dpi=600,
    bbox_inches="tight"
)


plt.savefig(
    "results/reporting_delay_by_country.tiff",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
df_wn["iso2"] = df_wn["iso_code"].apply(lambda x: pycountry.countries.get(alpha_3=x).alpha_2
                                                if pycountry.countries.get(alpha_3=x) else None) 

In [ ]:
df_wn.drop(columns=["iso_code"], inplace=True)

cols = df_wn.columns.tolist()
cols.remove("iso2")
idx = cols.index("outbreak_id")
cols.insert(idx + 1, "iso2")

df_wn = df_wn[cols]
df_wn.to_excel("data/wn_europe.xlsx",index=False)

# 2. EIOS extraction 

In [ ]:
import json
import pandas as pd
import time
from datetime import datetime, timedelta, timezone
import requests
from joblib import Parallel, delayed

In [ ]:
class EIOSClient:

    def __init__(
        self,
        base_url,
        api_version,
        tenant_id,
        client_id,
        client_secret,
        scope,
    ):
        self.base_url = base_url
        self.api_version = api_version
        self.filter_url = f"{base_url}/api/v{api_version}/Items/filter"

        self.tenant_id = tenant_id
        self.client_id = client_id
        self.client_secret = client_secret
        self.scope = scope

        self._token = None

    # Authorization
    def get_token(self):
        token_url = (
            f"https://login.microsoftonline.com/"
            f"{self.tenant_id}/oauth2/v2.0/token"
        )

        payload = {
            "grant_type": "client_credentials",
            "client_id": self.client_id,
            "client_secret": self.client_secret,
            "scope": self.scope,
        }

        r = requests.post(token_url, data=payload)
        r.raise_for_status()

        self._token = r.json()["access_token"]
        return self._token

    def headers(self):
        if not self._token:
            self.get_token()

        return {
            "Authorization": f"Bearer {self._token}",
            "Content-Type": "application/json",
        }

    # Filter items by country and disease
    def get_filtered_items(
        self,
        country_iso: str,
        disease: str,
        time_since: str,
        time_until: str | None = None,
        limit: int = 100,
    ):

        if time_until is None:
            time_until = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%S")

        all_items = []
        start = 0

        while True:

            query = {
                "rules": [
                    {
                        "name": "filterDate",
                        "operator": "dateRange",
                        "property": "processedOnDate",
                        "value": {
                            "gte": time_since,
                            "lte": time_until,
                        },
                    },
                    {
                        "operator": "in",
                        "property": "countriesIso",
                        "value": [country_iso],
                    },
                    {
                        "name": "categories-1",
                        "operator": "in",
                        "property": "categories",
                        "value": [f"cat:{disease}"],
                    },
                ],
                "groups": [
                    {
                        "name": "includeSourcesGroup",
                        "operator": "or",
                        "rules": [],
                        "groups": [
                            {
                                "name": "sourceFiltersGroup",
                                "operator": "and",
                                "rules": [
                                    {
                                        "name": "sourceSubjects",
                                        "operator": "in",
                                        "property": "source.subject",
                                        "value": [
                                            "General News",
                                            "Medical",
                                            "Agriculture",
                                            "European News",
                                            "Medical Official",
                                            "Financial News",
                                            "Official",
                                            "undefined",
                                            "Environment",
                                            "EU Institutions",
                                            "NGO",
                                            "Nuclear",
                                            "REC",
                                            "Technology",
                                            "Science",
                                        ],
                                    },
                                    {
                                        "name": "duplicatesSourceFiltersRule",
                                        "operator": "equals",
                                        "property": "isDuplicate",
                                        "value": False,
                                    },
                                ],
                                "groups": [],
                            }
                        ],
                    }
                ],
                "operator": "and",
                "start": start,
                "limit": limit,
                "sorts": [
                    {
                        "property": "processedOnDate",
                        "direction": "desc",
                    }
                ],
            }

            r = requests.post(self.filter_url, headers=self.headers(), json=query)
            r.raise_for_status()

            data = r.json()

            items = data.get("result", [])
            total = data.get("count", 0)

            all_items.extend(items)

            start += limit

            if start >= total:
                break

        return all_items

    # Get the full-text
    def get_full_text(self, article_id: str):
        full_text_url = f"{self.base_url}/api/v{self.api_version}/Items/{article_id}"

        r = requests.get(full_text_url, headers=self.headers())
        r.raise_for_status()

        res = r.json()
        return res

In [ ]:
# Import WAHIS data
df = pd.read_excel("data/wn_europe.xlsx")

In [ ]:
def load_config():
    with open("secrets/config.json", "r") as f:
        return json.load(f)
        
def fetch_country(client, country_iso, time_since, time_until):
    time.sleep(0.1)  
    try:
        items = client.get_filtered_items(
            country_iso=country_iso,
            disease="WestNileDisease",
            time_since=time_since,
            time_until=time_until
        )
        return items
    except Exception as e:
        print(f"Error fetching for {country_iso}: {e}")
        return []

In [ ]:
config = load_config()
       
client = EIOSClient(
    base_url=config["EIOS_BASE_URL"],
    api_version=config["EIOS_API_VERSION"],
    tenant_id=config["EIOS_TENANT_ID"],
    client_id=config["EIOS_CLIENT_ID"],
    client_secret=config["EIOS_CLIENT_SECRET"],
    scope=config["EIOS_SCOPE"]
)

In [ ]:
%%time
results = Parallel(n_jobs=4, prefer="threads")(
    delayed(fetch_country)(
        client,
        row["iso2"],
        row["outbreak_start_date"].strftime("%Y-%m-%d"),
        (row["outbreak_start_date"] + pd.Timedelta(days=7)).strftime("%Y-%m-%d"),
    )
    for _, row in df.iterrows()
)

In [ ]:
# Create a df and delete duplicates 
flat = [item for sublist in results for item in sublist]
results_df = pd.DataFrame(flat)
results_df = results_df.drop_duplicates(subset=["id"])

In [ ]:
results_df

In [ ]:
results_df.to_json("data/eios_historical.json", orient="records", indent=2)

# 3. TRANSLATE EIOS 

In [ ]:
import json
import pandas as pd
import numpy as np
from deep_translator import GoogleTranslator
from tqdm import tqdm
from joblib import Parallel, delayed

In [ ]:
def translate_eios_summary(df: pd.DataFrame) -> pd.DataFrame:
    """
    Takes a DataFrame of EIOS items, detects if they are non-English, 
    and translates the combined Title + Description to English.
    """
    df = df.copy()
    df.columns = df.columns.str.lower()

    # Clean strings
    df["title"] = df["title"].fillna("").astype(str)
    df["description"] = df["description"].fillna("").astype(str)

    # Combine raw text reliably
    df["original_text"] = (
        df["title"].str.strip() + " " + df["description"].str.strip()
    ).str.strip()

    # Normalize language ISO code
    df["languageiso"] = df["languageiso"].fillna("").str.lower()
    is_english = df["languageiso"].eq("en")

    # Set up baseline for English text
    df["en_text"] = np.where(is_english, df["original_text"], "")
    
    # Identify row masks needing translation
    mask_translate = (~is_english) & df["original_text"].ne("")

    if mask_translate.sum() > 0:
        translator = GoogleTranslator(source="auto", target="en")

        def safe_translate(text):
            if not isinstance(text, str) or not text.strip():
                return None
            try:
                return translator.translate(text)
            except Exception as e:
                print(f"Translation error: {str(e)[:120]}")
                return None

        # Apply progress bar tracking over rows
        tqdm.pandas(desc="Translating articles")
        df["translated_text"] = None
        df.loc[mask_translate, "translated_text"] = (
            df.loc[mask_translate, "original_text"].progress_apply(safe_translate)
        )

        # Fallback to translated text if English baseline wasn't set
        df["en_text"] = np.where(
            df["en_text"].ne(""),
            df["en_text"],
            df["translated_text"]
        )

    df["en_text"] = df["en_text"].fillna("").str.strip()

    return df


def parallel_translate_eios(df: pd.DataFrame, num_chunks, n_jobs: int = -1):
    
    num_chunks = min(len(df), num_chunks)
    
    indices = np.array_split(df.index, num_chunks)
    
    results = Parallel(n_jobs=n_jobs)(
        delayed(translate_eios_summary)(df.loc[idx]) for idx in indices
    )
    
    return pd.concat(results)

In [ ]:
with open("data/eios_historical.json", "r", encoding="utf-8") as f:
    raw_data = json.load(f)

df_raw = pd.DataFrame(raw_data)

df_en = parallel_translate_eios(df_raw, num_chunks=5, n_jobs=-1)

In [ ]:
df_en = df_en[["id", "pubdate", "en_text", "source", "eiosurl"]]

In [ ]:
output_file = "data/eios_historical_en.json"
df_en.to_json(output_file, orient="records", indent=2, force_ascii=False)

# 4. LLM ANALYSIS

In [ ]:
import os

# Create HF cache dirs
os.makedirs("/scratch/panelan/hf_cache", exist_ok=True)
os.makedirs("/scratch/panelan/hf_cache/hub", exist_ok=True)


# HuggingFace environment
os.environ["HF_HOME"] = "/scratch/panelan/hf_cache"
os.environ["HF_HUB_CACHE"] = "/scratch/panelan/hf_cache/hub"
os.environ["HF_HUB_DISABLE_XET"] = "1"


os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# Enforce deterministic scheduling for reproducibility
# https://docs.vllm.ai/en/latest/usage/reproducibility/
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

In [ ]:
# !nvidia-smi

In [ ]:
import subprocess
import random
import gc
import shutil
from tqdm.auto import tqdm
import torch
import pandas as pd
import matplotlib.pyplot as plt
import json
from datetime import datetime
from typing import List, Literal, Optional
from pydantic import BaseModel, Field
from vllm import LLM, SamplingParams
from vllm.sampling_params import GuidedDecodingParams

In [ ]:
# Load the data
with open("data/eios_historical_en.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [ ]:
llm_prompt = """

You are an epidemiologist classifying West Nile virus articles.

Return ONE JSON object.

Article:
{article_text}

---

SCOPE RULE (VERY IMPORTANT):
Only consider West Nile virus (WNV).

If the article is about ANY other disease → classify as "other".

---

CLASSIFICATION RULES (apply in order):

1) If there are NO human or animal infections mentioned
(only mosquito traps/pools/surveillance, no infected humans/animals, research on climate change etc):
→ label = "other"
→ outbreak_detected = false
→ species_affected = []
→ STOP

2) If the article is a surveillance report, statistics table, or seasonal summary:
→ label = "epi_summary"
→ outbreak_detected = false
*CRITERIA*: This includes reports capturing multi-national data, seasonal comparisons, or tracking cumulative statistics across multiple countries.
*EXAMPLE*: "WNV in Europe 2022: EU countries reported 292 cases across Italy (228), Greece (59), and Austria (2)." This is an epi_summary

3) If there are confirmed or suspected active, localized spikes, unexpected cases, or emergency notifications:
→ label = "outbreak_alert"
→ outbreak_detected = true
*CRITERIA*: Tone features real-time concern, unexpected increases, or immediate localized threats in a country/region.
*EXAMPLE*: "In recent weeks, increasingly alarming news has spread about the increase in cases of West Nile Disease in our country. The cases reported in Italy by the National Reference Center for WND, at the Zooprophylactic Institute, have risen to 230..." This must be an outbreak_alert with outbreak_detected = true.

1) If there are NO human or animal infections mentioned
(only mosquito traps/pools/surveillance, no infected humans/animals):
→ classify = "other"
→ outbreak_detected = false
→ species_affected = []
→ STOP

2) If the article is a surveillance report, statistics, or seasonal summary:
→ classify = "epi_summary"

3) If there are confirmed or suspected human/animal cases:
→ classify = "outbreak_alert"

---

CRITICAL RULES:
- outbreak_detected = true ONLY if the text explicitly mentions cases, infections or outbreak.
- If only research, modelling, risk, or discussion → outbreak_detected = false
- Mosquito-only positivity WITHOUT human/animal cases is ALWAYS "other" → outbreak_detected = false
- ONLY extract information explicitly stated in the text.
- Do NOT infer, assume, or generalize.
- Do NOT guess countries or species if not explicitly mentioned.
- If unsure, return empty list [] and false.
- If no specific event date is mentioned in the text, set "event_date" to null.
- Reason must be maximum 10 words, do not exceed 10 words under any circumstance

Return ONLY valid JSON.

JSON:"""

In [ ]:
# Pydantic schema 
class ArticleClassification(BaseModel):
    event_date: Optional[str] = Field(
        None, description="The event date mentioned in text (YYYY-MM-DD), or null if missing."
    )
    label: Literal["outbreak_alert", "epi_summary", "other"] = Field(
        description="Classification category based strictly on epidemiological host rules."
    )
    outbreak_detected: bool = Field(
        description="True ONLY if day_label is outbreak_alert, otherwise False."
    )
    countries: List[str] = Field(
        description="List of full country names explicitly found in the text."
    )
    iso2_codes: List[str] = Field(
        description="List of 2-letter ISO codes matching the identified countries."
    )
    species_affected: List[Literal["human", "horse", "bird", "other_animal"]] = Field(
        description="Array of host types impacted. Empty array [] if only mosquitoes test positive."
    )
    reasoning: str = Field(
        description="A strict brief justification, capped tightly under 10 words."
    )


# Convert Pydantic architecture directly to standard JSON Schema for vLLM
guided_schema = ArticleClassification.model_json_schema()

In [ ]:
guided_decoding_params = GuidedDecodingParams(json=guided_schema, backend="lm-format-enforcer")
sampling_params = SamplingParams(temperature=0.0, max_tokens=300,guided_decoding=guided_decoding_params)

In [ ]:
batch_size = 64  

llm_results = {}
all_outputs = []

In [ ]:
%%time

model_checkpoint =  "unsloth/Qwen2.5-14B-Instruct"

model_prompts = []
active_ids = []
    
for item in data:
    formatted_prompt = llm_prompt.format(
        article_text=item["en_text"]
    )
    model_prompts.append(formatted_prompt)
    active_ids.append(item["id"])
    
try: 
    # Load current model into GPU VRAM
    llm = LLM(
        model=model_checkpoint, 
        max_model_len=1024, 
        dtype="float16",
        enforce_eager=True,
        gpu_memory_utilization=0.85,
        trust_remote_code=True,
        download_dir=f"/scratch/panelan/hf_cache",
        seed = 8012026)
    for i in tqdm(range(0, len(model_prompts), batch_size), desc="Generating"):
        batch_prompts = model_prompts[i:i + batch_size]

        batch_outputs = llm.generate(batch_prompts, sampling_params)
        all_outputs.extend(batch_outputs)

    # Parse outputs
    for idx, output in enumerate(all_outputs):
        article_id = active_ids[idx]
        raw_output = output.outputs[0].text.strip()

        try:
            llm_results[article_id] = json.loads(raw_output)
        except json.JSONDecodeError:
            llm_results[article_id] = {
                "error": "Invalid JSON",
                "raw": raw_output
            }

    print(f"Completed extraction batch for {model_checkpoint}")

except Exception as e:
    print(f"Critical error executing {model_checkpoint}: {e}")

    for article_id in active_ids:
        llm_results[article_id] = {"error": str(e)}

finally:
    print("Flushing GPU VRAM")
    if "llm" in locals():
        del llm
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# Add info from the raw data 
lookup = {
     article["id"]: {
        "pub_date" : article.get("pubdate"),
        "url": article.get("source", {}).get("url"),
        "source_name": article.get("source", {}).get("name"),
    }
    for article in data
}


In [ ]:
for article_id, result in llm_results.items():
    info = lookup.get(article_id)
    result["pub_date"] = info.get("pub_date")
    result["url"] = info.get("url")
    result["source_name"] = info.get("source_name")

In [ ]:
# Check the first 3 items from the dictionary
for i, (key, value) in enumerate(list(llm_results.items())[:3]):
    print(f"=== Example {i+1} (Key: {key}) ===")
    print(json.dumps(value, indent=2, ensure_ascii=False))
    print("\n" + "-"*40 + "\n")

In [ ]:
# Save results
with open("results/llm_results_qwen2.5_14B.json", "w", encoding="utf-8") as f:
    json.dump(llm_results, f, indent=2, ensure_ascii=False)

# 5. LLM AS A JUDGE

This section is covered by the notebook LLM_as_a_judge.ipynb and was run in Google Colab.

# 6. ANALYSE GPT-O4 RESPONSES

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

In [ ]:
# Load the file and analyse the results
df = pd.read_json("results/gpt4o_evaluation_qwen2.5_14B.jsonl", lines=True)

print(len(df))

print("Average Quality Rating:", df['total_rating'].mean())

In [ ]:
df.columns

In [ ]:
df["label_and_outbreak_compliance"][0].get("status")

In [ ]:
status_cols = ["label_and_outbreak_compliance", "entity_grounding_compliance", "formatting_and_reasoning_compliance"]
df_status = df[status_cols].map(lambda x: x.get("status") if isinstance(x, dict) else None)
df_status_long = df_status.melt(var_name="rule", value_name="status")

In [ ]:
plt.figure(figsize=(9, 5))

ax = sns.countplot(
    data=df_status_long,
    x="rule",
    hue="status",
    palette="Set2"
)

# Add labels on bars
for container in ax.containers:
    ax.bar_label(container, fmt="%d", padding=2, fontsize=9)

ax.set_xticks(range(3))
ax.set_xticklabels(
    ["Classification", "Entity\ngrounding", "Formatting &\nreasoning"],
    fontsize=11
)

ax.set_title("Compliance status", fontsize=14, weight="bold")
ax.set_xlabel("")
ax.set_ylabel("Number of responses", fontsize=11)
ax.legend(title="Status", frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


plt.tight_layout()
plt.savefig(
    "results/compliance_status_qwen2.5_14B.pdf",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    "results/compliance_status_qwen2.5_14B.tiff",
    dpi=600,
    bbox_inches="tight"
)


plt.show()

In [ ]:
for col in df_status.columns:
    cnt = Counter(df_status[col])
    print(f"{col}: {cnt}")

In [ ]:
plt.figure(figsize=(9, 5))

ax = sns.countplot(
    data=df,
    x="total_rating", hue="total_rating",
    palette="Set2"
)

# Add labels on bars
for container in ax.containers:
    ax.bar_label(container, fmt="%d", padding=2, fontsize=9)

ax.set_xticks(range(3))
ax.set_xticklabels(
    ["Failure", "Correct", "Flawless"],
    fontsize=11
)

ax.set_title("Total rating", fontsize=14, weight="bold")
ax.set_xlabel("")
ax.set_ylabel("Number of responses", fontsize=11)
ax.legend(title="Rate", frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


plt.tight_layout()
plt.savefig(
    "results/total_rating_qwen2.5_14B.pdf",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    "results/total_rating_qwen2.5_14B.tiff",
    dpi=600,
    bbox_inches="tight"
)


plt.show()

In [ ]:
# Extract list of article ids that were correctly classified
article_ids = df.loc[df["total_rating"]==3].article_id
article_ids.to_csv("results/article_ids_highest_rate.csv", index=False, header=["article_id"])

# 7. CREATE A DATASET TO FINE-TUNE THE MODEL IN UNSLOTH (pt1)

In [ ]:
import pandas as pd
import json
from collections import Counter,defaultdict
import random
import ast

In [ ]:
# Articles that were rated 3
with open("results/article_ids_highest_rate.csv") as f:
    article_ids = pd.read_csv(f)

In [ ]:
len(article_ids)

In [ ]:
# Load the results
with open("results/llm_results_qwen2.5_14B.json", "r", encoding="utf-8") as f:
    llm_results = json.load(f)

In [ ]:
# Load the eios data
with open("data/eios_historical_en.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [ ]:
target_ids = set(article_ids["article_id"])

In [ ]:
# Create a balanced dataset 
grouped_by_label = defaultdict(list)

# Map target_ids into a quick-lookup dict for article content
article_lookup = {item["id"]: item for item in data if item["id"] in target_ids}

for article_id, result in llm_results.items():
    if article_id not in target_ids or article_id not in article_lookup:
        continue
        
    if isinstance(result, dict):
        label = result.get("label")
        if label:
            combined_record = result.copy()
            combined_record["article_id"] = article_id
            combined_record["article_text"] = article_lookup[article_id].get("en_text", "")
            
            combined_record.pop("pub_date", None)
            combined_record.pop("url", None)
            combined_record.pop("source_name", None)
            
            grouped_by_label[label].append(combined_record)

# Sample exactly up to 3 items for each label
sample_size = 3
balanced_rows = []

for label, records in grouped_by_label.items():
    num_to_sample = min(len(records), sample_size)
    
    sampled_records = random.sample(records, num_to_sample)
    balanced_rows.extend(sampled_records)
    
    print(f"Label '{label}': Sampled {num_to_sample} out of {len(records)} total records.")

df_balanced = pd.DataFrame(balanced_rows)
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

df_balanced.to_json(
    f"results/balanced_wnv_data_{sample_size}_per_label.json",
    orient="records",
    indent=4,
    force_ascii=False
)

# 7. CREATE A DATASET TO FINE-TUNE THE MODEL IN UNSLOTH (pt2)

In [ ]:
import os
import json
import time
import random
import pandas as pd
from openai import OpenAI

In [ ]:
with open("secrets/api-token-2026-06-29.txt", "r", encoding="utf-8") as f:
    api_token = f.read().strip()

In [ ]:
with open("results/balanced_wnv_data_3_per_label.json") as f:
     sample_data = json.load(f)

In [ ]:
sample_data

In [ ]:
client = OpenAI(
    api_key=api_token,
    base_url="https://api-gpt.jrc.ec.europa.eu/v1"
)

In [ ]:
system_prompt = f""""
You are generating a high-quality synthetic epidemiological dataset.

Your task has TWO phases:

1. Write ONE realistic news article or epidemiological report.
2. Annotate it with structured labels.

The output MUST be valid JSON.

=====================================================
TOPIC
=====================================================

The generated article may discuss West Nile Virus (WNV) OR another disease.

Generate all writing as if it were written by a real journalist, public health agency,
government bulletin, veterinary report, or epidemiological surveillance report.

The article should look authentic and naturally written.

Never mention that it is synthetic or AI-generated.

=====================================================
DIVERSITY REQUIREMENTS
=====================================================

Each sample must be independent and highly diverse.

Assume every generation will become one row of the final dataset.

Do not reuse wording, locations, organizations, dates, or statistics from previous examples.

Produce highly diverse articles.

Vary:

- publication style
- vocabulary
- sentence length
- article length
- country
- region
- publication source
- writing tone
- number of affected species
- chronology
- statistics
- quotes
- formatting style

Possible sources include:

- Ministry of Health
- Local newspaper
- National/ CDC /ECDC
- Veterinary authority
- WHO bulletin
- ECDC report
- Hospital report
- Agriculture department
- Municipality notice
- Research summary

Some articles should be:

- very short (70 words)
- medium (200 words)

Do NOT reuse templates.

Do NOT repeat phrasing.

=====================================================
CLASS BALANCING
=====================================================

Generate approximately:

33% other

33% epi_summary

34% outbreak_alert

=====================================================
SCOPE
=====================================================

Only West Nile Virus is relevant for classification.

If the article is primarily about another disease
(Dengue, Zika, Malaria, Influenza, COVID-19, etc.)

THEN

label = "other"

=====================================================
CLASSIFICATION RULES
=====================================================

Rule 1

If ONLY mosquito surveillance is discussed
(no infected humans or animals)

Examples:

- positive mosquito pools
- mosquito trapping
- vector surveillance
- larvae monitoring
- risk modelling
- climate discussion

THEN

label = "other"

outbreak_detected = false

species_affected = []

-----------------------------------------------------

Rule 2

If the article is a surveillance summary, weekly bulletin, annual report, seasonal overview or multi-country statistics

THEN

label = "epi_summary"

outbreak_detected = false

Examples:

- Annual surveillance report
- Weekly epidemiological bulletin
- Europe reported 292 WNV cases in 2022
- Historical comparison

-----------------------------------------------------

Rule 3

If the article reports active confirmed or suspected infections,localized transmission, unexpected increases, or public health alerts

THEN

label = "outbreak_alert"

outbreak_detected = true

Examples:

- New human infections
- Horse infections
- Bird mortality linked to WNV
- Regional outbreak notification

=====================================================
EXTRACTION RULES
=====================================================

Extract ONLY information explicitly stated.

Never infer.

Never guess.

If information is missing, return:

[]

or

null

Countries must be full names.

ISO codes must correspond exactly.

Species may only contain:

- human
- horse
- bird
- other_animal

Reasoning must contain fewer than 10 words.

Event date:

Use the event date explicitly mentioned.

If absent:

null

====================================================
SOME REAL EXAMPLES
====================================================
Study the following examples carefully.

They illustrate the writing style and the expected
annotation.

Do NOT copy them.

Do NOT reuse names, numbers, locations,
dates or wording.

Generate a completely new article.

{sample_data}

=====================================================
OUTPUT FORMAT
=====================================================

Return ONLY valid JSON.

- article_id should be a string with a combination of letters and numbers
- If no specific event date is mentioned in the text, set "event_date" to null.

{{"article_id": "...",
"article_text": "...",
"event_date": "...",
"label": "...",
"outbreak_detected": true,
"countries": [],
"iso2_codes": [],
"species_affected": [],
"reasoning": "..."}}

"""

In [ ]:
countries_pool = [
    # Americas
    "Brazil", "Canada", "Colombia", "Haiti", "Mexico", "Peru", "United States",
    # Europe
    "France", "Germany", "Italy", "Norway", "Poland", "Spain", "United Kingdom",
    # Africa
    "Democratic Republic of the Congo", "Egypt", "Ethiopia", "Kenya", "Nigeria", "South Africa", "Senegal",
    # Asia & Middle East
    "Bangladesh", "India", "Indonesia", "Japan", "Saudi Arabia", "Thailand", "Vietnam",
    # Oceania
    "Australia", "Fiji", "Papua New Guinea"
]


In [ ]:
output_file = "results/synthetic_data.jsonl"

target_records = 1000
records_per_batch = 10

existing_records = 0
if os.path.exists(output_file):
    with open(output_file, "r") as f:
        existing_records = sum(1 for line in f if line.strip())
    print(f"Resuming generation. Found {existing_records} existing records in {output_file}.")
    
batches_needed = (target_records - existing_records) // records_per_batch

with open(output_file, "a", encoding="utf-8") as f:
    for batch_idx in range(batches_needed):
        batch_countries = random.sample(countries_pool, k=records_per_batch)
        countries_str = ", ".join(batch_countries)

        batch_instruction = f"""

        Generate exactly 10 distinct synthetic epidemiological records. 
        
        CRITICAL GEOGRAPHIC CONSTRAINT:
        Each of the 10 records must be set in a different country, you can select one of the following countries:
        {json.dumps(batch_countries)}

        NOTES: you can also not to mention any country at all!
        
        """
        try:
            chat_completion = client.chat.completions.create(
                model="gpt-oss-120b",
                messages=[
                    {
                        "role": "system",
                        "content": system_prompt
                    },
                    {
                        "role": "user",
                        "content": batch_instruction
                    }
                ],
                response_format={"type": "json_object"},
                temperature=0.9,
                stream=False,
                timeout=60.0
            )
            
            raw_content = chat_completion.choices[0].message.content
            parsed = json.loads(raw_content)
            
            records_list = parsed.get("records", [])
            
            # Write to JSONL
            for record in records_list:
                f.write(json.dumps(record) + "\n")
            f.flush()
            
            print(f"Saved {len(records_list)} geographically diverse records.\n")
            
        except json.JSONDecodeError:
            print(" Failed: Invalid JSON returned. Skipping this batch.\n")
            continue
        except Exception as e:
            print(f"Error: {e}\n")
            time.sleep(5)
            continue

In [ ]:
synthetic_data = []
# Import the data 
with open("results/synthetic_data.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip(): 
            record = json.loads(line)
            synthetic_data.append(record)

In [ ]:
llm_prompt = """

You are an epidemiologist classifying West Nile virus articles.

Return ONE JSON object.

Article:
{article_text}

---

SCOPE RULE (VERY IMPORTANT):
Only consider West Nile virus (WNV).

If the article is about ANY other disease → classify as "other".

---

CLASSIFICATION RULES (apply in order):

1) If there are NO human or animal infections mentioned
(only mosquito traps/pools/surveillance, no infected humans/animals, research on climate change etc):
→ label = "other"
→ outbreak_detected = false
→ species_affected = []
→ STOP

2) If the article is a surveillance report, statistics table, or seasonal summary:
→ label = "epi_summary"
→ outbreak_detected = false
*CRITERIA*: This includes reports capturing multi-national data, seasonal comparisons, or tracking cumulative statistics across multiple countries.
*EXAMPLE*: "WNV in Europe 2022: EU countries reported 292 cases across Italy (228), Greece (59), and Austria (2)." This is an epi_summary

3) If there are confirmed or suspected active, localized spikes, unexpected cases, or emergency notifications:
→ label = "outbreak_alert"
→ outbreak_detected = true
*CRITERIA*: Tone features real-time concern, unexpected increases, or immediate localized threats in a country/region.
*EXAMPLE*: "In recent weeks, increasingly alarming news has spread about the increase in cases of West Nile Disease in our country. The cases reported in Italy by the National Reference Center for WND, at the Zooprophylactic Institute, have risen to 230..." This must be an outbreak_alert with outbreak_detected = true.

1) If there are NO human or animal infections mentioned
(only mosquito traps/pools/surveillance, no infected humans/animals):
→ classify = "other"
→ outbreak_detected = false
→ species_affected = []
→ STOP

2) If the article is a surveillance report, statistics, or seasonal summary:
→ classify = "epi_summary"

3) If there are confirmed or suspected human/animal cases:
→ classify = "outbreak_alert"

---

CRITICAL RULES:
- outbreak_detected = true ONLY if the text explicitly mentions cases, infections or outbreak.
- If only research, modelling, risk, or discussion → outbreak_detected = false
- Mosquito-only positivity WITHOUT human/animal cases is ALWAYS "other" → outbreak_detected = false
- ONLY extract information explicitly stated in the text.
- Do NOT infer, assume, or generalize.
- Do NOT guess countries or species if not explicitly mentioned.
- If unsure, return empty list [] and false.
- If no specific event date is mentioned in the text, set "event_date" to null.
- Reason must be maximum 10 words, do not exceed 10 words under any circumstance

Return ONLY valid JSON.

JSON:"""

In [ ]:
synthetic_data

In [ ]:
# Save the dataset for fine-tuning  (Alpaca instruction format)
ft_dataset = []

for item in synthetic_data:
    article_id = item["article_id"]
    # Generate the prompt
    formatted_llm_prompt = llm_prompt.format(
        article_text=item["article_text"]
    )
    
    # Get the response directly from llm_results
    metadata = {k: v for k, v in item.items() if k not in ("article_id", "article_text")}
    
    ft_dataset.append({
        "article_id": article_id,
        "instruction": formatted_llm_prompt,  
        "input": "",                         
        "output": str(metadata)                
    })

In [ ]:
df_synthetic_data = pd.DataFrame(synthetic_data)
print(df_synthetic_data["label"].value_counts())

In [ ]:
ft_dataset[0]

In [ ]:
with open("results/ft_dataset.json", "w", encoding="utf-8") as f:
    json.dump(ft_dataset, f, indent=4, ensure_ascii=False)

# 8. FINE TUNE THE MODEL WITH UNSLOTH

This dataset was used to fine-tune Qwen2.5-14B-Instruct in unsloth using Google Colab. To do so, I followed the guidelines provided in the [unsloth blog](https://unsloth.ai/blog). The code is available in finetune_qwen2.5.ipynb. 

# 9. RUN THE FINE-TUNED MODEL WITH VLLM

In [ ]:
import os

# Create HF cache dirs
os.makedirs("/scratch/panelan/hf_cache", exist_ok=True)
os.makedirs("/scratch/panelan/hf_cache/hub", exist_ok=True)


# HuggingFace environment
os.environ["HF_HOME"] = "/scratch/panelan/hf_cache"
os.environ["HF_HUB_CACHE"] = "/scratch/panelan/hf_cache/hub"
os.environ["HF_HUB_DISABLE_XET"] = "1"


os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# Enforce deterministic scheduling for reproducibility
# https://docs.vllm.ai/en/latest/usage/reproducibility/
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

In [ ]:
import subprocess
import random
import gc
import shutil
from tqdm.auto import tqdm
import torch
import pandas as pd
import matplotlib.pyplot as plt
import json
from datetime import datetime
from typing import List, Literal, Optional
from pydantic import BaseModel, Field
from vllm import LLM, SamplingParams
from vllm.sampling_params import GuidedDecodingParams

In [ ]:
# Load the data
with open("data/eios_historical_en.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [ ]:
llm_prompt = """

You are an epidemiologist classifying West Nile virus articles.

Return ONE JSON object.

Article:
{article_text}

---

SCOPE RULE (VERY IMPORTANT):
Only consider West Nile virus (WNV).

If the article is about ANY other disease → classify as "other".

---

CLASSIFICATION RULES (apply in order):

1) If there are NO human or animal infections mentioned
(only mosquito traps/pools/surveillance, no infected humans/animals, research on climate change etc):
→ label = "other"
→ outbreak_detected = false
→ species_affected = []
→ STOP

2) If the article is a surveillance report, statistics table, or seasonal summary:
→ label = "epi_summary"
→ outbreak_detected = false
*CRITERIA*: This includes reports capturing multi-national data, seasonal comparisons, or tracking cumulative statistics across multiple countries.
*EXAMPLE*: "WNV in Europe 2022: EU countries reported 292 cases across Italy (228), Greece (59), and Austria (2)." This is an epi_summary

3) If there are confirmed or suspected active, localized spikes, unexpected cases, or emergency notifications:
→ label = "outbreak_alert"
→ outbreak_detected = true
*CRITERIA*: Tone features real-time concern, unexpected increases, or immediate localized threats in a country/region.
*EXAMPLE*: "In recent weeks, increasingly alarming news has spread about the increase in cases of West Nile Disease in our country. The cases reported in Italy by the National Reference Center for WND, at the Zooprophylactic Institute, have risen to 230..." This must be an outbreak_alert with outbreak_detected = true.

1) If there are NO human or animal infections mentioned
(only mosquito traps/pools/surveillance, no infected humans/animals):
→ classify = "other"
→ outbreak_detected = false
→ species_affected = []
→ STOP

2) If the article is a surveillance report, statistics, or seasonal summary:
→ classify = "epi_summary"

3) If there are confirmed or suspected human/animal cases:
→ classify = "outbreak_alert"

---

CRITICAL RULES:
- outbreak_detected = true ONLY if the text explicitly mentions cases, infections or outbreak.
- If only research, modelling, risk, or discussion → outbreak_detected = false
- Mosquito-only positivity WITHOUT human/animal cases is ALWAYS "other" → outbreak_detected = false
- ONLY extract information explicitly stated in the text.
- Do NOT infer, assume, or generalize.
- Do NOT guess countries or species if not explicitly mentioned.
- If unsure, return empty list [] and false.
- If no specific event date is mentioned in the text, set "event_date" to null.
- Reason must be maximum 10 words, do not exceed 10 words under any circumstance

Return ONLY valid JSON.

JSON:"""

In [ ]:
# Pydantic schema
class ArticleClassification(BaseModel):
    event_date: Optional[str] = Field(
        None, description="The event date mentioned in text (YYYY-MM-DD), or null if missing."
    )
    label: Literal["outbreak_alert", "epi_summary", "other"] = Field(
        description="Classification category based strictly on epidemiological host rules."
    )
    outbreak_detected: bool = Field(
        description="True ONLY if day_label is outbreak_alert, otherwise False."
    )
    countries: List[str] = Field(
        description="List of full country names explicitly found in the text."
    )
    iso2_codes: List[str] = Field(
        description="List of 2-letter ISO codes matching the identified countries."
    )
    species_affected: List[Literal["human", "horse", "bird", "other_animal"]] = Field(
        description="Array of host types impacted. Empty array [] if only mosquitoes test positive."
    )
    reasoning: str = Field(
        description="A strict brief justification, capped tightly under 10 words."
    )


# Convert Pydantic architecture directly to standard JSON Schema for vLLM
guided_schema = ArticleClassification.model_json_schema()

In [ ]:
guided_decoding_params = GuidedDecodingParams(json=guided_schema, backend="lm-format-enforcer")
sampling_params = SamplingParams(temperature=0.0, max_tokens=300,guided_decoding=guided_decoding_params)

In [ ]:
batch_size = 64  

llm_results = {}
all_outputs = []

In [ ]:
%%time

model_checkpoint =  "anjelinejeline/Qwen2.5-14B-Instruct-epi"

model_prompts = []
active_ids = []
    
for item in data:
    formatted_prompt = llm_prompt.format(
        article_text=item["en_text"]
    )
    model_prompts.append(formatted_prompt)
    active_ids.append(item["id"])
    
try: 
    # Load current model into GPU VRAM
    llm = LLM(
        model=model_checkpoint, 
        max_model_len=1024, 
        dtype="float16",
        enforce_eager=True,
        gpu_memory_utilization=0.85,
        trust_remote_code=True,
        download_dir=f"/scratch/panelan/hf_cache",
        seed = 8012026)
    for i in tqdm(range(0, len(model_prompts), batch_size), desc="Generating"):
        batch_prompts = model_prompts[i:i + batch_size]

        batch_outputs = llm.generate(batch_prompts, sampling_params)
        all_outputs.extend(batch_outputs)

    # Parse outputs
    for idx, output in enumerate(all_outputs):
        article_id = active_ids[idx]
        raw_output = output.outputs[0].text.strip()

        try:
            llm_results[article_id] = json.loads(raw_output)
        except json.JSONDecodeError:
            llm_results[article_id] = {
                "error": "Invalid JSON",
                "raw": raw_output
            }

    print(f"Completed extraction batch for {model_checkpoint}")

except Exception as e:
    print(f"Critical error executing {model_checkpoint}: {e}")

    for article_id in active_ids:
        llm_results[article_id] = {"error": str(e)}

finally:
    print("Flushing GPU VRAM")
    if "llm" in locals():
        del llm
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# Add info from the raw data 
lookup = {
     article["id"]: {
        "pub_date": article.get("pubdate"),
        "url": article.get("source", {}).get("url"),
        "source_name": article.get("source", {}).get("name"),
    }
    for article in data
}


In [ ]:
# Save results
with open("results/llm_results_ft_qwen2.5_14B_epi.json", "w", encoding="utf-8") as f:
    json.dump(llm_results, f, indent=2, ensure_ascii=False)

# 10. LLLM AS A JUDGE FOR THE FINE TUNED MODEL

This section is covered by the notebook LLM_as_a_judge.ipynb and was run in GoogleColab.

# 11. COMPARE GPTO-4 EVALUATIONS

In [ ]:
import pandas as pd 
import json 
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [ ]:
# EIOS data
with open("data/eios_historical_en.json", "r", encoding="utf-8") as f:
    eios_json = json.load(f)

df_eios = pd.DataFrame(eios_json)

In [ ]:
# LLMs output 
# Load LLM 1 results
with open("results/llm_results_qwen2.5_14B.json", "r", encoding="utf-8") as f:
    llm1_results = json.load(f)

df_llm1 = pd.DataFrame(
    [{"article_id": key, **value} for key, value in llm1_results.items()]
)

# Load LLM 2 results
with open(
    "results/llm_results_ft_qwen2.5_14B_epi.json", "r", encoding="utf-8"
) as f:
    llm2_results = json.load(f)

df_llm2 = pd.DataFrame(
    [{"article_id": key, **value} for key, value in llm2_results.items()]
)

In [ ]:
# Gpt-4o assessment 
df1_gpt = pd.read_json("results/gpt4o_evaluation_qwen2.5_14B.jsonl", lines=True)
df2_gpt = pd.read_json("results/gpt4o_evaluation_ft_qwen2.5_14B_epi.jsonl", lines=True)

In [ ]:
df1_gpt.columns

In [ ]:
status_cols = ["label_and_outbreak_compliance", "entity_grounding_compliance", "formatting_and_reasoning_compliance"]
def preprocess_eval_df(df, model_suffix):
    df_clean = df[status_cols].map(lambda x: x.get("status") if isinstance(x, dict) else None)
    
    df_clean["total_rating"] = pd.to_numeric(df["total_rating"], errors='coerce')
    
    df_clean["article_id"] = df["article_id"]
    
    rename_dict = {col: f"{col}_{model_suffix}" for col in status_cols}
    rename_dict["total_rating"] = f"total_rating_{model_suffix}."
    
    df_clean = df_clean.rename(columns={
        **{col: f"{col}_{model_suffix}" for col in status_cols},
        "total_rating": f"total_rating_{model_suffix}"
    })
    
    return df_clean
df1_gpt_clean = preprocess_eval_df(df1_gpt, "m1")
df2_gpt_clean = preprocess_eval_df(df2_gpt, "m2")

df_compared = pd.merge(df1_gpt_clean, df2_gpt_clean, on="article_id", how="inner")

In [ ]:
len(df_compared)

In [ ]:
# Which model performed better?
df_long = df_compared.melt(
    id_vars=["article_id"], 
    var_name="metric_model", 
    value_name="status"
)


df_long[["metric", "model"]] = df_long["metric_model"].str.rsplit("_", n=1, expand=True)

# Calculate the pass rates
pass_rates = df_long.groupby(["metric", "model", "status"]).size().unstack(fill_value=0)
pass_rates["pass_rate_%"] = (pass_rates["Pass"] / (pass_rates["Pass"] + pass_rates["Fail"])) * 100
print(pass_rates)

In [ ]:
# Look at 1 example

# What is West Nile Virus, what are the symptoms and how to prevent it? 
# National | 16:10 hrs In humans it presents with fever, general malaise and occasionally 
# with serious symptoms such as encephalitis or meningitis (inflammation of the brain). 
# Although human infections in WON-endemic areas are common, most of these are generally mild or subclinical, while the disease...

df_long.loc[df_long["article_id"]=="excelsior-c824e444853a8b14ab1d600036f1116d"]

In [ ]:
# Focus on the cases where the fine-tuned model assigned a wrong label 
ids_fail_m2 = df_long.loc[(df_long["metric_model"] == "label_and_outbreak_compliance_m2") & (df_long["status"] == "Fail")]["article_id"]

In [ ]:
df2_gpt_fail = df2_gpt.loc[df2_gpt["article_id"].isin(list(ids_fail_m2))][["article_id","label_and_outbreak_compliance"]]
print(df2_gpt_fail)

In [ ]:
df2_gpt_fail["comment"] = df2_gpt_fail["label_and_outbreak_compliance"].apply(
    lambda x: x["comment"]
)

In [ ]:
df2_fail_all = df2_gpt_fail.merge(df_llm2, on = "article_id")

In [ ]:
df2_fail_all = df2_fail_all.merge(df_eios, left_on = "article_id", right_on = "id")

In [ ]:
# Check the comments
df2_fail_all["comment"].unique()

In [ ]:
# Check the 2 cases where the comment does not follow the normal pattern  
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
print(df2_fail_all.loc[df2_fail_all["comment"] == "Incorrect country; should be Montana, USA."])
print("#"*100)
print(df2_fail_all.loc[df2_fail_all["comment"] == "Label is incorrect; EEE is mentioned."])

In [ ]:
# Standardize the comments 
def standardize_comment(comment):

    m = re.search(r"should be '([^']+)'", comment)
    if m:
        return f"Expected {m.group(1)}"
    if comment == "Label 'other' is incorrect; outbreak detected.":
        return "Expected outbreak_alert"
    else:
        return comment 


df2_fail_all["comment_st"] = df2_fail_all["comment"].apply(standardize_comment)

In [ ]:
df2_fail_all["comment_st"].unique()

In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.histplot(
    data=df2_fail_all,
    y="label",
    hue="comment_st",
    multiple="stack",
    palette="Set2",
    shrink=0.8,
    edgecolors="none"
)


for c in ax.containers:
    labels = [f"{int(w)}" if (w := p.get_width()) > 0 else "" for p in c]
    ax.bar_label(c, labels=labels, label_type="center", color="black", fontsize=9)

ax.set(title="Qwen2.5-14B-Instruct-epi\nlabel classification errors", ylabel="", xlabel="Count")
ax.title.set_weight("bold")
ax.title.set_fontsize(13)

ax.set_ylabel("Count", fontsize=11)
ax.tick_params(axis="x", labelsize=10)


sns.despine() 
sns.move_legend(ax, "upper left", bbox_to_anchor=(1.02, 1), frameon=False, title="Failure comment")


plt.savefig(
    "results/label_failure_m2.pdf",
    dpi=600,
    bbox_inches="tight"
)
plt.savefig(
    "results/label_failure_m2.tiff",
    dpi=600,
    bbox_inches="tight"
)


plt.tight_layout()

In [ ]:
# Compare total rating Model 1 and 2 
print(df_compared[["total_rating_m1", "total_rating_m2"]].describe())

In [ ]:
df_long.columns

In [ ]:
# Plot the status difference between the 2 models 
df_plot = df_long[
    df_long["metric"] != "total_rating"
][["metric", "status", "model"]].copy()

df_plot = df_plot.astype(str)

df_plot["metric"] = df_plot["metric"].replace({
    "entity_grounding_compliance": "Entity\ngrounding",
    "formatting_and_reasoning_compliance": "Formatting &\nreasoning",
    "label_and_outbreak_compliance": "Label &\noutbreak"
})

model_labels = {
    "m1": "Qwen2.5-14B-Instruct",
    "m2": "Qwen2.5-14B-Instruct-epi"
}

df_plot["model_label"] = df_plot["model"].replace(model_labels)

g = sns.catplot(
    data=df_plot,
    kind="count",
    x="metric",
    hue="status",
    col="model_label",
    palette="Set2",
    height=4.5,
    aspect=1.1
)


for ax in g.axes.flat:
    for container in ax.containers:
        ax.bar_label(
            container,
            fmt="%d",
            padding=2,
            fontsize=9
        )

    ax.set_xlabel("")
    ax.set_ylabel("Number of responses")
    ax.tick_params(axis="x", labelsize=10)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


g.set_titles("{col_name}", fontsize=13, weight="bold")

g.figure.suptitle(
    "Compliance evaluation",
    fontsize=15,
    weight="bold",
    y=1.05
)

g._legend.set_title("Status")
g._legend.set_frame_on(False)

plt.tight_layout()

plt.savefig(
    "results/compliance_status_m1_m2.pdf",
    dpi=600,
    bbox_inches="tight"
)
plt.savefig(
    "results/compliance_status_m1_m2.tiff",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Check label disagreement between the 2 models
pd.crosstab(
    df_compared["label_and_outbreak_compliance_m1"],
    df_compared["label_and_outbreak_compliance_m2"]
)

In [ ]:
df_disagreement = df_compared[
    df_compared["label_and_outbreak_compliance_m1"] != 
    df_compared["label_and_outbreak_compliance_m2"]
].copy()

In [ ]:
df_compared["label_and_outbreak_compliance_m1"].unique()
df_compared["label_and_outbreak_compliance_m2"].unique()

In [ ]:
len(df_disagreement)

In [ ]:
len(df_disagreement)/1781

In [ ]:
discrepancy_ids = df_disagreement["article_id"].tolist()

In [ ]:
df_llm1_dis = df_llm1[df_llm1["article_id"].isin(discrepancy_ids)]
df_llm2_dis = df_llm2[df_llm2["article_id"].isin(discrepancy_ids)]

In [ ]:
df_discrepancies = pd.merge(df_llm1_dis,df_llm2_dis, on = "article_id", suffixes=("_m1","_m2"))

In [ ]:
df1_gpt = df1_gpt.rename(
    columns={col: f"{col}_m1" for col in df1_gpt.columns if col != "article_id"}
)

In [ ]:
df_discrepancies = pd.merge(df_discrepancies, df1_gpt, on="article_id")

In [ ]:
df2_gpt = df2_gpt.rename(
    columns={col: f"{col}_m2" for col in df2_gpt.columns if col != "article_id"}
)

In [ ]:
df_discrepancies = pd.merge(df_discrepancies, df2_gpt, on="article_id")

In [ ]:
df_discrepancies = pd.merge(df_discrepancies, df_eios, left_on="article_id", right_on = "id")

In [ ]:
len(df_discrepancies)

In [ ]:
df_discrepancies.columns

In [ ]:
df_discrepancies.to_excel("results/df_discrepancies.xlsx", index=False)

In [ ]:
# Plot the total rating of the 2 models 
df_rating = df_long[
    df_long["metric"] == "total_rating"
][["status", "model"]].copy()

palette = {
    "Qwen2.5-14B-Instruct": "#7F7F7F",      # Baseline (gray)
    "Qwen2.5-14B-Instruct-epi": "#4C72B0"   # Fine-tuned model (blue)
}


model_labels = {
    "m1": "Qwen2.5-14B-Instruct",
    "m2": "Qwen2.5-14B-Instruct-epi"
}

df_rating["model_label"] = df_rating["model"].replace(model_labels)

df_rating = df_rating.astype(str)

# Force ordering
rating_order = ["1", "2", "3"]


plt.figure(figsize=(7, 5))

ax = sns.countplot(
    data=df_rating,
    x="status",
    hue="model_label",
    order=rating_order,
    palette=palette
)


for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%d",
        padding=2,
        fontsize=10
    )


ax.set_title(
    "Overall quality rating",
    fontsize=14,
    weight="bold"
)

ax.set_xlabel(
    "Rating assigned"
)

ax.set_ylabel(
    "Number of responses"
)

ax.legend(
    title="Model",
    frameon=False
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


plt.tight_layout()

plt.savefig(
    "results/total_rating_m1_m2.pdf",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    "results/total_rating_m1_m2.tiff",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

# 12. DAILY ALERTS (FINE-TUNED MODEL)

In [ ]:
import time
import json
import pandas as pd
import pycountry
from joblib import Parallel, delayed
from tqdm import tqdm
from deep_translator import GoogleTranslator
import requests

In [ ]:
class EIOSClient:

    def __init__(
        self,
        base_url,
        api_version,
        tenant_id,
        client_id,
        client_secret,
        scope,
    ):
        self.base_url = base_url
        self.api_version = api_version
        self.filter_url = f"{base_url}/api/v{api_version}/Items/filter"

        self.tenant_id = tenant_id
        self.client_id = client_id
        self.client_secret = client_secret
        self.scope = scope

        self._token = None

    # Authorization
    def get_token(self):
        token_url = (
            f"https://login.microsoftonline.com/"
            f"{self.tenant_id}/oauth2/v2.0/token"
        )

        payload = {
            "grant_type": "client_credentials",
            "client_id": self.client_id,
            "client_secret": self.client_secret,
            "scope": self.scope,
        }

        r = requests.post(token_url, data=payload)
        r.raise_for_status()

        self._token = r.json()["access_token"]
        return self._token

    def headers(self):
        if not self._token:
            self.get_token()

        return {
            "Authorization": f"Bearer {self._token}",
            "Content-Type": "application/json",
        }

    # Filter items by country and disease
    def get_filtered_items(
        self,
        country_iso: str,
        disease: str,
        time_since: str,
        time_until: str | None = None,
        limit: int = 100,
    ):

        if time_until is None:
            time_until = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%S")

        all_items = []
        start = 0

        while True:

            query = {
                "rules": [
                    {
                        "name": "filterDate",
                        "operator": "dateRange",
                        "property": "processedOnDate",
                        "value": {
                            "gte": time_since,
                            "lte": time_until,
                        },
                    },
                    {
                        "operator": "in",
                        "property": "countriesIso",
                        "value": [country_iso],
                    },
                    {
                        "name": "categories-1",
                        "operator": "in",
                        "property": "categories",
                        "value": [f"cat:{disease}"],
                    },
                ],
                "groups": [
                    {
                        "name": "includeSourcesGroup",
                        "operator": "or",
                        "rules": [],
                        "groups": [
                            {
                                "name": "sourceFiltersGroup",
                                "operator": "and",
                                "rules": [
                                    {
                                        "name": "sourceSubjects",
                                        "operator": "in",
                                        "property": "source.subject",
                                        "value": [
                                            "General News",
                                            "Medical",
                                            "Agriculture",
                                            "European News",
                                            "Medical Official",
                                            "Financial News",
                                            "Official",
                                            "undefined",
                                            "Environment",
                                            "EU Institutions",
                                            "NGO",
                                            "Nuclear",
                                            "REC",
                                            "Technology",
                                            "Science",
                                        ],
                                    },
                                    {
                                        "name": "duplicatesSourceFiltersRule",
                                        "operator": "equals",
                                        "property": "isDuplicate",
                                        "value": False,
                                    },
                                ],
                                "groups": [],
                            }
                        ],
                    }
                ],
                "operator": "and",
                "start": start,
                "limit": limit,
                "sorts": [
                    {
                        "property": "processedOnDate",
                        "direction": "desc",
                    }
                ],
            }

            r = requests.post(self.filter_url, headers=self.headers(), json=query)
            r.raise_for_status()

            data = r.json()

            items = data.get("result", [])
            total = data.get("count", 0)

            all_items.extend(items)

            start += limit

            if start >= total:
                break

        return all_items

    # Get the full-text
    def get_full_text(self, article_id: str):
        full_text_url = f"{self.base_url}/api/v{self.api_version}/Items/{article_id}"

        r = requests.get(full_text_url, headers=self.headers())
        r.raise_for_status()

        res = r.json()
        return res

In [ ]:
# Load the results
with open("results/llm_results_ft_qwen2.5_14B_epi.json", "r", encoding="utf-8") as f:
    llm_results = json.load(f)

In [ ]:
def load_config():
    with open("secrets/config.json", "r") as f:
        return json.load(f)

config = load_config()
       
client = EIOSClient(
    base_url=config["EIOS_BASE_URL"],
    api_version=config["EIOS_API_VERSION"],
    tenant_id=config["EIOS_TENANT_ID"],
    client_id=config["EIOS_CLIENT_ID"],
    client_secret=config["EIOS_CLIENT_SECRET"],
    scope=config["EIOS_SCOPE"]
)

In [ ]:
def fetch_full_text(client,article_id:str):
    time.sleep(0.1)
    try:
        res = client.get_full_text(article_id=article_id)
        item = {
            article_id: res["fullText"]
            }
        return item
        
    except Exception as e:
        print(f"Error fetching for {article_id}: {e}")
        return []

In [ ]:
full_texts = Parallel(n_jobs=4, prefer="threads")(
    delayed(fetch_full_text)(client=client, article_id=article_id)
    for article_id in llm_results.keys()
)

full_texts_dict= {}

for d in full_texts:
    full_texts_dict.update(d)

In [ ]:
# Translate the full_text in english 
translator = GoogleTranslator(source="auto", target="en")

def translate_text(text):
    if text is None or str(text).strip() == "":
        return ""
    try:
        return translator.translate(str(text))
    except Exception as e:
        return str(text)



# The lambda returns: (text_id, {"original": text, "translated": translated_text})
translated_dict = dict(
    Parallel(n_jobs=-1, prefer="threads")(
        delayed(lambda text_id, txt: (text_id, {
            "original": txt, 
            "translated": translate_text(txt)
        }))(text_id, text)
        for text_id, text in tqdm(full_texts_dict.items(), desc="Parallel Translation")
    )
)

In [ ]:
#  Inject  full-texts 
for article_id, article_data in llm_results.items():
    # Grab the text data if it exists for this ID
    text_data = translated_dict.get(article_id)
    
    if text_data:
        article_data["full_text"] = text_data["original"]
        article_data["full_text_en"] = text_data["translated"]
    else:
        # Fallbacks if an ID failed to fetch or translate
        article_data["full_text"] = ""
        article_data["full_text_en"] = ""

In [ ]:
# Define a function to aggregate the data at the day level 
def daily_aggregation(raw_data):
    aggregated_alerts = {}
    
    for article_id, info in raw_data.items():
        if info.get("outbreak_detected"):
            if info.get("pub_date"):
                pub_day = info["pub_date"].split('T')[0]
            else:
                pub_day = "unknown date"
            
            countries = info.get("countries", [])
            iso_codes = info.get("iso2_codes", [])  
            
            if not countries:
                countries = ["unknown"]
                iso_codes = ["UNK"]
            else:
                while len(iso_codes) < len(countries):
                    iso_codes.append("UNK")

            species_list = info.get("species_affected", [])
            if not species_list:
                species_list = ["unknown"]

            reason = info.get("reasoning", "No reasoning provided by LLM.")
            source_name = info.get("source_name","")
            url = info.get("url","")
            original_text = info.get("full_text", "")
            translated_text = info.get("full_text_en", "")
           
        
            for c, iso in zip(countries, iso_codes):
                for s in species_list:
                    alert_key = f"{pub_day}_{c}_{s}"

                    # Initialize the alert structure if it's the first time we see this key
                    if alert_key not in aggregated_alerts:
                        aggregated_alerts[alert_key] = {
                            "pub_date": pub_day,
                            "country": c,
                            "iso2_code": iso,
                            "species": s,       
                            "article_count": 0,
                            "article_ids": [],
                            "evidence_reasonings": {},
                            "source_name":{},
                            "url":{},
                            "full_text": {},           
                            "full_text_en": {},
                            
                            
                        }

                    current_alert = aggregated_alerts[alert_key]

                    if article_id not in current_alert["article_ids"]:
                        
                        # Add tracking statistics
                        current_alert["article_count"] += 1
                        current_alert["article_ids"].append(article_id)
                        current_alert["evidence_reasonings"][article_id] = reason
                        current_alert["source_name"][article_id] = source_name
                        current_alert["url"][article_id] = url
                        current_alert["full_text"][article_id] = original_text
                        current_alert["full_text_en"][article_id] = translated_text
                 
                        
    return list(aggregated_alerts.values())

In [ ]:
alerts = daily_aggregation(llm_results)

In [ ]:
df_alerts = pd.DataFrame(alerts)

In [ ]:
df_alerts.to_json("results/alerts_epi.json", orient="records", indent=2, force_ascii=False)

# 13. ASSESS RELEVANCE FOR WOAH (FINE-TUNED MODEL)

In [ ]:
import pandas as pd 
import numpy as np
import ast
import json
import re
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

In [ ]:
with open("results/alerts_epi.json", "r", encoding="utf-8") as f:
    alerts = json.load(f)

In [ ]:
df_wahis = pd.read_excel("data/wn_europe.xlsx")

In [ ]:
df_alerts = pd.DataFrame(alerts)

In [ ]:
df_alerts[~(df_alerts["species"].isin(["human","unknown"]))]

In [ ]:
def match_with_temporal_window(
    official: pd.DataFrame,
    news: pd.DataFrame,
    official_geo_col: str,
    news_geo_col: str,
    official_species_col: str,
    news_species_col: str,
    outbreak_start_col: str,
    days: int,
    pub_date_col: str
) -> pd.DataFrame:
    """
    Find correspondence between an official data and open-source news .

    Parameters:
    -----------
    official : DataFrame containing official outbreak logs.
    news : DataFrame containing open-source epidemic intelligence.
    official_geo_col : Name of country/geography column in official dataset
    (e.g., 'iso2')
    news_geo_col : Name of country/geography column in news dataset (e.g.,
    'iso2_code')
    official_species_col : Name of species column in official dataset
    news_species_col : Name of species column in news dataset
    outbreak_start_col : Name of start date column in official dataset
    days : Number of days to consider since the outbreak start date
    pub_date_col : Name of publication date column in news dataset
    """
    # Harmonize matching 
    official["_normalized_start"] = pd.to_datetime(official[outbreak_start_col],format="%Y-%m-%d")
    news["_normalized_pub"] = pd.to_datetime(news[pub_date_col],format="%Y-%m-%d")

    official["_normalized_geo"] = (
        official[official_geo_col].astype(str).str.upper().str.strip()
    )
    news["_normalized_geo"] = (
        news[news_geo_col].astype(str).str.upper().str.strip()
    )

    official["_normalized_species"] = (
        official[official_species_col].astype(str).str.lower().str.strip()
    )
    news["_normalized_species"] = (
        news[news_species_col].astype(str).str.lower().str.strip()
    )

    # Merge on the newly created normalized tracking keys
    merged = pd.merge(
        official, news, on=["_normalized_geo", "_normalized_species"], how="inner",
        suffixes = ("_off","_news")
    )

    # Execute window logic based on the dynamic timeline of each match
    window_start = merged["_normalized_start"] 
    window_end = window_start + pd.Timedelta(days=days)

    in_window_mask = (merged["_normalized_pub"] >= window_start) & (
        merged["_normalized_pub"] <= window_end
    )
    results = merged[in_window_mask].copy()

    internal_keys = [
        "_normalized_start",
        "_normalized_report",
        "_normalized_pub",
        "_normalized_geo",
        "_normalized_species",
    ]
    results = results.drop(columns=internal_keys, errors="ignore")

    return results

In [ ]:
matched_data = match_with_temporal_window(
    official=df_wahis,
    news=df_alerts,
    official_geo_col="iso2",  
    news_geo_col="iso2_code", 
    official_species_col="species",
    news_species_col="species",
    outbreak_start_col="outbreak_start_date",
    days=7,
    pub_date_col="pub_date"
)

In [ ]:
print("Type:", matched_data["outbreak_start_date"].dtype)
print("Type:", matched_data["pub_date"].dtype)

In [ ]:
len(matched_data)

In [ ]:
matched_data["pub_date"] = pd.to_datetime(
    matched_data["pub_date"], format="%Y-%m-%d"
).astype("datetime64[us]")

In [ ]:
matched_data.head(2)

In [ ]:
matched_data["query"] = matched_data.apply(
    lambda row: (
        f"Outbreak of  West Nile Fever in {row['level1_name']}, {row['country_off']}."
        f"Affected species: {row['species_off']}."
    ),
    axis=1,
)

In [ ]:
model_checkpoint = "sentence-transformers/all-MiniLM-L6-v2"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModel.from_pretrained(model_checkpoint)
model.eval()

In [ ]:
# Mean pooling
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

    return (
        torch.sum(token_embeddings * input_mask_expanded, dim=1)
        / torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
    )

In [ ]:
# Embedding function 
def encode(texts):
    """
    texts: string or list of strings
    returns: normalized embeddings
    """
    if isinstance(texts, str):
        texts = [texts]

    encoded = tokenizer(
        texts,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

    with torch.no_grad():
        output = model(**encoded)

    embeddings = mean_pooling(output, encoded["attention_mask"])
    embeddings = F.normalize(embeddings, p=2, dim=1)

    return embeddings

In [ ]:
# Split full-text into sentence 
def split_sentences(text):
    if pd.isna(text):
        return []

    return [
        s.strip()
        for s in re.split(r'(?<=[.!?])\s+', text)
        if len(s.strip()) > 20
    ]

In [ ]:
# Retrieve the best matching sentence
def retrieve_best_sentence(query, article):

    sentences = split_sentences(article)

    if len(sentences) == 0:
        return None, None

    query_emb = encode(query)
    sentence_embs = encode(sentences)

    similarities = torch.mm(query_emb, sentence_embs.T).squeeze(0)

    best_idx = similarities.argmax().item()

    return (
        sentences[best_idx],
        similarities[best_idx].item()
    )

In [ ]:
def analyze_all_articles_in_row(row):
    query = row["query"]
    text_dict = row["full_text_en"]
    
    # Placeholders to store the results for this row
    best_sentences = {}
    similarities = {}
    
    # Ensure text_dict is a valid dictionary before looping
    if isinstance(text_dict, dict):
        for article_id, text in text_dict.items():
            if text and str(text).strip() != "":
                # Calculate similarity for this specific article's text string
                sentence, score = retrieve_best_sentence(query, text)
                
                # Store the results keyed by the article_id
                best_sentences[article_id] = sentence
                similarities[article_id] = score
            else:
                best_sentences[article_id] = ""
                similarities[article_id] = 0.0
                
    # Return both results as individual dictionary structures
    return pd.Series({
        "best_sentences": best_sentences,
        "similarities": similarities
    })

In [ ]:
# Apply to the matched_data to generate two new columns
matched_data[["best_sentences", "similarities"]] = matched_data.apply(
    analyze_all_articles_in_row, 
    axis=1
)

In [ ]:
# Create the mask to include articles published before reporting day
before_report_mask = matched_data["pub_date"] < matched_data["first_reporting_date"]

filtered_matched_data = matched_data[before_report_mask].copy()

filtered_matched_data["days_from_pub_to_report"] = (
     filtered_matched_data["pub_date"] - filtered_matched_data["first_reporting_date"] 
).dt.days

In [ ]:
len(filtered_matched_data)

In [ ]:
filtered_matched_data.to_excel("results/matched_wahis_eios_validation_epi.xlsx", index=False)

In [ ]:
filtered_matched_data

In [ ]:
filtered_matched_data["outbreak_id"].unique() # 28 unique outbreak_id

In [ ]:
len(filtered_matched_data) 

In [ ]:
print(filtered_matched_data["similarities"].iloc[0])

In [ ]:
filtered_matched_data = pd.read_excel("results/matched_wahis_eios_validation_epi.xlsx")

In [ ]:
# Check the similarity scores 
sim_dicts = filtered_matched_data["similarities"].apply(ast.literal_eval)
all_scores = [score for d in sim_dicts for score in d.values()]

In [ ]:
summary = {
    "count": len(all_scores),
    "mean": np.mean(all_scores),
    "median": np.median(all_scores),
    "std": np.std(all_scores),
    "min": np.min(all_scores),
    "max": np.max(all_scores),
}

print(summary)


In [ ]:
all_ids =[key for d in sim_dicts for key in d.keys()]

In [ ]:
len(all_ids)

In [ ]:
len(set(all_ids))

# 14. CHECK RELEVANCE FOR HUMAN DATA

Data on human cases were downloaded using Google Colab because access to external websites is blocked in the BDAPP.
The corresponding notebook is download_human_data.ipynb.

In [ ]:
import pandas as pd 
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Official cases
df_cases_it = pd.read_excel("data/italy_wn_2018_2025.xlsx")
df_ecdc = pd.read_excel("data/ecdc_wn_2018_2023.xlsx")

In [ ]:
# LLM results
df_llm = pd.read_json("/eos/jeodpp/home/users/panelan/wn_alert/results/alerts.jsonl", lines=True)

In [ ]:
# Compare the signal against the official italian data
# Can LLMEpidemic Tracker signals be useful for now-casting?
df_llm_it = df_llm.loc[(df_llm["iso2_code"] == "IT") & (df_llm["species"] == "human")]
df_cases_it["data"] = pd.to_datetime(df_cases_it["data"])
df_llm_it["pub_date"] = pd.to_datetime(df_llm_it["pub_date"])

In [ ]:
df_cases_it = df_cases_it.sort_values("data").reset_index(drop=True)
df_llm_it = df_llm_it.sort_values("pub_date").reset_index(drop=True)

In [ ]:
# Calculate LLM signal 7 days before each bulletin excluding the previous bulletin day

llm_signal = []

for i, bulletin_date in enumerate(df_cases_it["data"]):

    # Start of surveillance interval
    if i == 0:
        start = bulletin_date - pd.Timedelta(days=7)
    else:
        start = df_cases_it.loc[i-1, "data"] + pd.Timedelta(days=1)

    # Bulletin includes information available until previous day
    end = bulletin_date - pd.Timedelta(days=1)

    signal = df_llm_it.loc[
        (df_llm_it["pub_date"] >= start) &
        (df_llm_it["pub_date"] <= end),
        "article_count"
    ].sum()

    llm_signal.append(signal)


df_cases_it["llm_articles_current_interval"] = llm_signal

In [ ]:
# Now-casting correlation

rho, p = spearmanr(
    df_cases_it["llm_articles_current_interval"],
    df_cases_it["new_cases"]
)

print("rho:", rho)
print("p-value:", p)

In [ ]:
# Pick a bulletin index 
i = 1

bulletin_date = df_cases_it.loc[i, "data"]

previous_bulletin = df_cases_it.loc[i - 1, "data"]

start = previous_bulletin + pd.Timedelta(days=1)
end = bulletin_date - pd.Timedelta(days=1)

print("Previous bulletin:", previous_bulletin.date())
print("Current bulletin :", bulletin_date.date())
print("Interval         :", start.date(), "to", end.date())

In [ ]:
sns.set_theme(style="white")

fig, ax1 = plt.subplots(figsize=(12, 5))

cases_color = "#264653"   
llm_color = "#E76F51"    

# Official weekly cases
ax1.bar(
    df_cases_it["data"],
    df_cases_it["new_cases"],
    width=5,
    color=cases_color,
    edgecolor="none",
    label="New WNV cases"
)

ax1.set_ylabel("Count", fontsize=11)
ax1.set_xlabel("")
ax1.tick_params(axis="both", labelsize=10)


# LLMEpidemic Tracker signal
ax2 = ax1.twinx()

ax2.plot(
    df_cases_it["data"],
    df_cases_it["llm_articles_current_interval"],
    color=llm_color,
    linewidth=2.2,
    marker="o",
    markersize=3.5,
    label="LLMEpidemic Tracker signal"
)


ax2.set_ylabel("")
ax2.set_yticks([])
ax2.spines["right"].set_visible(False)


# Spearman correlation
rho = 0.716  

ax1.text(
    1.02,
    0.75,
    f"Spearman's ρ = {rho:.2f}\n$p$ < 0.001",
    transform=ax1.transAxes,
    fontsize=10,
    va="top",
    ha="left"
)


handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    handles1 + handles2,
    labels1 + labels2,
    loc="upper left",
    bbox_to_anchor=(1.02, 1),
    frameon=False,
    fontsize=10
)



ax1.set_title(
    "LLMEpidemic Tracker signal vs official data",
    fontsize=13,
    fontweight="bold"
)



sns.despine(ax=ax1)
sns.despine(ax=ax2, left=True, right=True)


plt.tight_layout()

plt.savefig(
    "results/LLMEpidemicTracker_nowcasting_italy.pdf",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    "results/LLMEpidemicTracker_nowcasting_italy.tiff",
    dpi=600,
    bbox_inches="tight"
)


plt.show()

In [ ]:
# Can LLMEpidemic Tracker provide an earlier indication relative to the first human WNV case reported to ECDC??

In [ ]:
df_ecdc

In [ ]:
df_ecdc["first_case_date"] = pd.to_datetime(df_ecdc["first_case_date"])

# Country-year first human case
df_ecdc_first = (
    df_ecdc
    .groupby(["year", "country_name", "country_iso"])
    .agg(
        ecdc_first_case_date=("first_case_date", "min"),
        ecdc_total_cases=("human_cases", "sum"),
        ecdc_confirmed_cases=("confirmed_human_cases", "sum")
    )
    .reset_index()
)

df_ecdc_first.head()

In [ ]:
# Filter the LLM signal
df_llm_human = df_llm[
    df_llm["species"] == "human"
].copy()

df_llm_human["pub_date"] = pd.to_datetime(df_llm_human["pub_date"])

countries = [
    "Italy",
    "Greece",
    "Hungary",
    "Germany",
    "Spain"
]

df_llm_human = df_llm_human[
    df_llm_human["country"].isin(countries)
]

In [ ]:
# Find the first LLMEpidemic Tracker human signal
df_llm_first = (
    df_llm_human
    .groupby(
        [
            df_llm_human["pub_date"].dt.year,
            "country"
        ]
    )
    .agg(
        llm_first_signal_date=("pub_date", "min")
    )
    .reset_index()
)

df_llm_first.rename(
    columns={"pub_date": "year"},
    inplace=True
)

In [ ]:
df_early_detection = df_ecdc_first.merge(
    df_llm_first,
    left_on=["year", "country_name"],
    right_on=["year", "country"],
    how="inner"
)

df_early_detection

In [ ]:
df_early_detection["lead_days"] = (
    df_early_detection["ecdc_first_case_date"] -
    df_early_detection["llm_first_signal_date"]
).dt.days

In [ ]:
n_total = len(df_early_detection)
n_before = (df_early_detection["lead_days"] > 0).sum()
n_after = (df_early_detection["lead_days"] < 0).sum()
n_long = (df_early_detection["lead_days"] > 100).sum()

print(f"Before ECDC: {n_before}/{n_total}")
print(f"After ECDC: {n_after}/{n_total}")
print(f">100 days: {n_long}/{n_total}")

In [ ]:
# Create classification of signals
df_early_detection["signal_class"] = "Within plausible window"

df_early_detection.loc[
    df_early_detection["lead_days"] > 100,
    "signal_class"
] = "Long lead time (>100 days)"

df_early_detection.loc[
    df_early_detection["lead_days"] < 0,
    "signal_class"
] = "After ECDC first case"


df_plot = (
    df_early_detection
    .sort_values("lead_days")
    .copy()
)

df_plot["country_year"] = (
    df_plot["country_name"]
    + " "
    + df_plot["year"].astype(str)
)


plt.figure(figsize=(10, 7))

ax = sns.barplot(
    data=df_plot,
    y="country_year",
    x="lead_days",
    hue="signal_class",
    dodge=False,
    palette={
        "Within plausible window": "#264653",
        "Long lead time (>100 days)": "#E76F51",
        "After ECDC first case": "#B0B0B0"
    },
    edgecolor="none"
)

plt.axvline(0, color="black", linewidth=1)

plt.xlabel(
    "Lead time (days)",
    fontsize=11
)

plt.ylabel("Country-year", fontsize=11)

plt.title(
    "LLMEpidemic Tracker first signals\nand ECDC first reported human WNV cases",
    fontsize=13,
    fontweight="bold"
)

plt.legend(
    title="Signal classification",
    frameon=False,
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

sns.despine()

plt.tight_layout()

plt.savefig(
    "results/LLMEpidemicTracker_ECDC_lead_time.pdf",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    "results/LLMEpidemicTracker_ECDC_lead_time.tiff",
    dpi=600,
    bbox_inches="tight"
)


plt.show()